In [ ]:
from contextlib import contextmanager
from datetime import datetime
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
import mlflow
import optuna
import pandas as pd
import seaborn as sns
import xgboost as xgb
from autogluon.tabular import TabularDataset, TabularPredictor
from dotenv import load_dotenv
from mlflow import MlflowClient
from mlflow.models import infer_signature
from mrmr import mrmr_classif
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    fbeta_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score

load_dotenv("../config/.env")
mlflow.set_registry_uri("databricks-uc")
optuna.logging.set_verbosity(optuna.logging.WARNING)

#### Handicap Result Prediction

In [ ]:
client = MlflowClient()

In [ ]:
df = pd.read_parquet("../data/processed/jleague.parquet")
df_train = df.query("fold == 'train'")
df_test = df.query("fold == 'development'")

resp_var = "hcap_res"
exp_vars = [
    "rate_h2h_win", "rate_h2h_lose", "hcap_mag", "n_rest_day_net",
    "rate_seas_win_net", "rate_seas_lose_net", "scores_net", "rank_net",
    "rating_seas_net", "rating_hist_net", "xg_net", "xg_sup",
    "xg_h2h_win", "xg_h2h_lose",
]

X_train, y_train = df_train[exp_vars], df_train[resp_var]
X_test, y_test = df_test[exp_vars], df_test[resp_var]

In [ ]:
# feature selection
exp_vars = mrmr_classif(X=X_train, y=y_train, K=10)

# anomaly detection
model = IsolationForest(contamination=0.15, random_state=42)
is_outlier = model.fit_predict(df_train[exp_vars]) == 0

In [ ]:
def evaluate_model(y_true, y_hat, pi_hat):
    return {
        "accuracy": accuracy_score(y_true, y_hat),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_hat),
        "f05_score": fbeta_score(y_true, y_hat, beta=0.5),
        "f1_score": fbeta_score(y_true, y_hat, beta=1.0),
        "f2_score": fbeta_score(y_true, y_hat, beta=2.0),
        "precision": precision_score(y_true, y_hat),
        "recall": recall_score(y_true, y_hat),
        "log_loss": log_loss(y_true, pi_hat),
        "roc_auc": roc_auc_score(y_true, pi_hat),
        "mcc": matthews_corrcoef(y_true, y_hat),
    }


def report_clf(y_true, y_hat):
    return classification_report(
        y_true,
        y_hat,
        target_names=["Away", "Home"],
        digits=4,
    )


@contextmanager
def plot_confusion_matrix(y_true, y_hat):
    fig, ax = plt.subplots()
    try:
        cm = confusion_matrix(y_true, y_hat)
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
        ax.set_title("Confusion Matrix")
        ax.set_xticks([0.5, 1.5])
        ax.set_xticklabels(["Away", "Home"])
        ax.set_yticks([0.5, 1.5])
        ax.set_yticklabels(["Away", "Home"])
        yield fig
    finally:
        plt.close(fig)


@contextmanager
def plot_feature_importance(ft, importance):
    fig, ax = plt.subplots()
    try:
        ft_import = pd.DataFrame({"feature": ft, "importance": importance}) \
            .sort_values(by="importance", ascending=True)
        ax.barh(ft_import["feature"], ft_import["importance"])
        ax.set_xlabel("Importance")
        ax.set_title("Feature Importance")
        fig.tight_layout()
        yield fig
    finally:
        plt.close(fig)

##### (1) Logistic Regression (Baseline)

In [ ]:
params = {
    "max_iter": 1_000,
    "random_state": 42,
}

In [ ]:
dt = datetime.now().strftime("%Y%m%d%H%M")

with mlflow.start_run(run_name=f"logistic_regression_{dt}") as run:
    mlflow.set_tag("model_family", "glm")

    clf = LogisticRegression(**params)
    clf.fit(X_train[~is_outlier][exp_vars], y_train[~is_outlier])

    y_hat = clf.predict(X_test[exp_vars])
    pi_hat = clf.predict_proba(X_test[exp_vars])[:, 1]

    mlflow.log_params(params)

    mlflow.log_dict({"features": exp_vars}, "features.json")

    metrics = evaluate_model(y_test, y_hat, pi_hat)
    mlflow.log_metrics(metrics)

    report = report_clf(y_test, y_hat)
    mlflow.log_text(report, "classification_report.txt")

    with plot_confusion_matrix(y_test, y_hat) as fig:
        mlflow.log_figure(fig, "plots/confusion_matrix.png")

    info = mlflow.sklearn.log_model(
        sk_model=clf,
        registered_model_name="betsim.models.logistic_regression",
        signature=infer_signature(
            X_test[exp_vars],
            pd.DataFrame({"y_hat": y_hat, "pi_hat": pi_hat}),
        ),
        input_example=X_test[exp_vars].head(),
        name="logistic_regression",
    )

In [ ]:
client.set_registered_model_alias(
    name="betsim.models.logistic_regression",
    version=info.registered_model_version,
    alias="champion",
)

##### (2) XGBoost w/ Optuna

In [ ]:
base_params = {
    "eval_metric": "logloss",
    "n_jobs": 1,
    "objective": "binary:logistic",
    "random_state": 42,
}

def objective(trial):
    params_trial = {
        **base_params,
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 1e-8, 5.0, log=True),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
    }

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    scores = cross_val_score(
        xgb.XGBClassifier(**params_trial),
        X_train[exp_vars],
        y_train,
        cv=cv,
        scoring="neg_log_loss",
        n_jobs=-1,
    )

    return -scores.mean()


study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="xgboost_optuna_tuning",
)
study.optimize(objective, n_trials=50, show_progress_bar=True)

params = {**base_params, **study.best_params}

In [ ]:
dt = datetime.now().strftime("%Y%m%d%H%M")

with mlflow.start_run(run_name=f"xgboost_optuna_{dt}"):
    mlflow.set_tag("model_family", "xgboost")

    clf = xgb.XGBClassifier(**params)
    clf.fit(X_train[exp_vars], y_train, verbose=False)

    y_hat = clf.predict(X_test[exp_vars])
    pi_hat = clf.predict_proba(X_test[exp_vars])[:, 1]

    mlflow.log_params(params)

    mlflow.log_dict({"features": exp_vars}, "features.json")

    metrics = evaluate_model(y_test, y_hat, pi_hat)
    mlflow.log_metrics(metrics)

    report = report_clf(y_test, y_hat)
    mlflow.log_text(report, "classification_report.txt")

    with plot_confusion_matrix(y_test, y_hat) as fig:
        mlflow.log_figure(fig, "plots/confusion_matrix.png")

    with plot_feature_importance(exp_vars, clf.feature_importances_) as fig:
        mlflow.log_figure(fig, "plots/feature_importance.png")

    info = mlflow.xgboost.log_model(
        xgb_model=clf,
        registered_model_name="betsim.models.xgboost_optuna",
        signature=infer_signature(
            X_test[exp_vars],
            pd.DataFrame({"y_hat": y_hat, "pi_hat": pi_hat}),
        ),
        input_example=X_test[exp_vars].head(),
        name="xgboost_optuna",
    )

In [ ]:
client.set_registered_model_alias(
    name="betsim.models.xgboost_optuna",
    version=info.registered_model_version,
    alias="champion",
)

##### (3) AutoGluon

In [ ]:
base_params = {
    "label": resp_var,
    "problem_type": "binary",
    "eval_metric": "log_loss",
    "verbosity": 0,
}

params = {
    "num_bag_folds": 5,
    "num_bag_sets": 1,
    "num_stack_levels": 1,
    "presets": "best_v150",
    "time_limit": 60 * 5,
}

X_train = TabularDataset(pd.concat([X_train[exp_vars], y_train], axis=1))
X_test = TabularDataset(pd.concat([X_test[exp_vars], y_test], axis=1))

In [ ]:
class AutoGluonPyFuncModel(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        pth_model = context.artifacts["ag_predictor"]
        self.model = TabularPredictor.load(pth_model)

    def predict(self, context, model_input: pd.DataFrame):
        return self.model.predict_proba(model_input)

In [ ]:
dt = datetime.now().strftime("%Y%m%d%H%M")

with mlflow.start_run(run_name=f"autogluon_best_v150_{dt}"):
    mlflow.set_tag("model_family", "autogluon")

    with TemporaryDirectory() as pth_tmp:
        clf = TabularPredictor(**base_params, path=pth_tmp)
        clf.fit(X_train, **params, excluded_model_types=["NN_TORCH", "FASTAI"])

        pi_hat = clf.predict_proba(X_test)[1]
        y_hat = (pi_hat >= 0.5).astype(int)

        mlflow.log_params({**base_params, **params})

        mlflow.log_dict({"features": exp_vars}, "features.json")

        metrics = evaluate_model(y_test, y_hat, pi_hat)
        mlflow.log_metrics(metrics)

        report = report_clf(y_test, y_hat)
        mlflow.log_text(report, "classification_report.txt")

        leaderboard = clf.leaderboard(X_test, silent=True)
        mlflow.log_table(leaderboard, "leaderboard_test.json")

        with plot_confusion_matrix(y_test, y_hat) as fig:
            mlflow.log_figure(fig, "plots/confusion_matrix.png")

        info = mlflow.pyfunc.log_model(
            artifact_path="autogluon_best_v150",
            python_model=AutoGluonPyFuncModel(),
            artifacts={"ag_predictor": pth_tmp},
            registered_model_name="betsim.models.autogluon_best_v150",
            signature=infer_signature(
                X_test,
                pd.DataFrame({"y_hat": y_hat, "pi_hat": pi_hat})
            ),
            input_example=X_test.head(),
        )

In [ ]:
client.set_registered_model_alias(
    name="betsim.models.autogluon_best_v150",
    version=info.registered_model_version,
    alias="champion",
)